# Presentation Tables

This notebook builds the presentation tables directly and logs the best result for each metric under every table.

In [1]:
from pathlib import Path
import re
import sys

import pandas as pd
from IPython.display import display

pd.options.display.float_format = "{:.3f}".format

CWD = Path.cwd().resolve()
REPO_ROOT = next((path for path in [CWD, *CWD.parents] if (path / "inference_eval").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {CWD}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from inference_eval.analysis.metrics_analysis import (
    DEFAULT_RUN_DIR,
    PRIMARY_METRIC_LABELS,
    build_summary_table,
    load_metrics,
    prompt_sort_key,
)

C3_DEEP_DIVE_METRICS = [
    "C3 Code Acc",
    "Emotion Precision",
    "Emotion Recall",
    "Emotion F1",
    "Tier Accuracy",
]
MODEL_DISPLAY_ORDER = {
    "Gemma 3 4B": 1,
    "Qwen 3 8B": 2,
    "GPT-5-nano": 3,
    "GPT-5.4-mini": 4,
}
PROMPT_DISPLAY = {
    "majority_class": "Baseline",
    "zero_shot": "Zero-shot",
    "few_shot": "Few-shot",
    "retrieval_few_shot": "Retrieval few-shot",
}
DISPLAY_PROMPT_ORDER = {
    "Baseline": 0,
    "Zero-shot": 1,
    "Few-shot": 2,
    "Retrieval few-shot": 3,
}


def _model_display_name(provider: str, model: str) -> str:
    if provider == "baseline":
        return "Majority baseline"
    if model.startswith("gemma3:"):
        return f"Gemma 3 {model.split(':', 1)[1].upper()}"
    if model.startswith("qwen3:"):
        return f"Qwen 3 {model.split(':', 1)[1].upper()}"
    dated_openai = re.match(r"^(gpt-[a-z0-9.-]+)-\d{4}-\d{2}-\d{2}$", model)
    if dated_openai:
        model = dated_openai.group(1)
    if model == "gpt-5-nano":
        return "GPT-5-nano"
    if model == "gpt-5.4-mini":
        return "GPT-5.4-mini"
    return model


def _retrieval_backend_label(variant_name: str, prompt_strategy: str) -> str | None:
    if prompt_strategy != "retrieval_few_shot":
        return None
    if "sentence-transformer" in variant_name or "minilm" in variant_name:
        return "MiniLM"
    return "TF-IDF"


def _prompt_display_name(variant_name: str, prompt_strategy: str) -> str:
    if prompt_strategy != "retrieval_few_shot":
        return PROMPT_DISPLAY.get(prompt_strategy, prompt_strategy)
    backend = _retrieval_backend_label(variant_name, prompt_strategy)
    return f"Retrieval few-shot ({backend})"


def _strategy_rollup_name(prompt_strategy: str) -> str:
    return PROMPT_DISPLAY.get(prompt_strategy, prompt_strategy)


def _with_display_columns(summary_df: pd.DataFrame) -> pd.DataFrame:
    display_df = summary_df.copy()
    display_df["Model"] = [
        _model_display_name(provider, model)
        for provider, model in zip(display_df["provider"], display_df["model"], strict=False)
    ]
    display_df["Prompt Strategy"] = [
        _prompt_display_name(variant_name, prompt_strategy)
        for variant_name, prompt_strategy in zip(
            display_df["variant_name"],
            display_df["prompt_strategy"],
            strict=False,
        )
    ]
    display_df["model_order"] = display_df["Model"].map(MODEL_DISPLAY_ORDER).fillna(999)
    display_df["prompt_order"] = [
        prompt_sort_key(prompt_strategy)[0] for prompt_strategy in display_df["prompt_strategy"]
    ]
    return display_df


def _load_summary(run_dir: str | Path = DEFAULT_RUN_DIR) -> pd.DataFrame:
    metrics_df = load_metrics(Path(run_dir))
    return build_summary_table(metrics_df)


def build_main_results_table(run_dir: str | Path = DEFAULT_RUN_DIR) -> pd.DataFrame:
    summary_df = _with_display_columns(_load_summary(run_dir))
    summary_df = summary_df[summary_df["prompt_strategy"] != "majority_class"].copy()
    table = summary_df.sort_values(["prompt_order", "model_order", "Prompt Strategy", "Model"]).reset_index(drop=True)
    return table[["Model", "Prompt Strategy", *PRIMARY_METRIC_LABELS]].copy()


def _collapse_strategy_variants(summary_df: pd.DataFrame) -> pd.DataFrame:
    collapsed = _with_display_columns(summary_df)
    collapsed = collapsed[collapsed["prompt_strategy"] != "majority_class"].copy()
    collapsed = (
        collapsed.groupby(["Model", "prompt_strategy"], dropna=False)[PRIMARY_METRIC_LABELS]
        .mean()
        .reset_index()
    )
    collapsed["Prompt Strategy"] = collapsed["prompt_strategy"].map(_strategy_rollup_name)
    collapsed["model_order"] = collapsed["Model"].map(MODEL_DISPLAY_ORDER).fillna(999)
    collapsed["prompt_order"] = [
        prompt_sort_key(prompt_strategy)[0] for prompt_strategy in collapsed["prompt_strategy"]
    ]
    return collapsed


def build_prompt_strategy_table(run_dir: str | Path = DEFAULT_RUN_DIR) -> pd.DataFrame:
    collapsed = _collapse_strategy_variants(_load_summary(run_dir))
    table = (
        collapsed.groupby(["prompt_strategy", "Prompt Strategy"], dropna=False)
        .agg(
            models_averaged=("Model", "nunique"),
            **{metric: (metric, "mean") for metric in PRIMARY_METRIC_LABELS},
        )
        .reset_index()
        .sort_values("prompt_strategy", key=lambda column: column.map(lambda value: prompt_sort_key(value)[0]))
        .reset_index(drop=True)
    )
    table = table.rename(columns={"models_averaged": "Models Averaged"})
    return table[["Prompt Strategy", "Models Averaged", *PRIMARY_METRIC_LABELS]].copy()


def build_family_comparison_table(run_dir: str | Path = DEFAULT_RUN_DIR) -> pd.DataFrame:
    collapsed = _collapse_strategy_variants(_load_summary(run_dir))
    table = (
        collapsed.groupby("Model", dropna=False)
        .agg(
            prompt_strategies=(
                "Prompt Strategy",
                lambda values: ", ".join(sorted(set(values), key=lambda value: DISPLAY_PROMPT_ORDER.get(value, 999))),
            ),
            **{metric: (metric, "mean") for metric in PRIMARY_METRIC_LABELS},
        )
        .reset_index()
    )
    table["model_order"] = table["Model"].map(MODEL_DISPLAY_ORDER).fillna(999)
    table = table.sort_values(["model_order", "Model"]).reset_index(drop=True)
    table = table.rename(columns={"prompt_strategies": "Prompt Strategies"})
    return table[["Model", "Prompt Strategies", *PRIMARY_METRIC_LABELS]].copy()


def build_c3_deep_dive_table(run_dir: str | Path = DEFAULT_RUN_DIR, *, top_n: int = 3) -> pd.DataFrame:
    metrics_df = load_metrics(Path(run_dir))
    summary_df = _with_display_columns(build_summary_table(metrics_df))
    summary_df = summary_df[summary_df["prompt_strategy"] != "majority_class"].copy()
    c3_df = metrics_df[metrics_df["construct"] == "c3"].copy()
    c3_pivot = (
        c3_df.pivot_table(index="variant_name", columns="metric", values="value", aggfunc="first").reset_index()
    )
    metadata = summary_df[["variant_name", "Model", "Prompt Strategy", "model_order", "prompt_order"]].drop_duplicates()
    table = metadata.merge(c3_pivot, on="variant_name", how="inner")
    table = table.rename(
        columns={
            "code_accuracy": "C3 Code Acc",
            "emotion_label_precision": "Emotion Precision",
            "emotion_label_recall": "Emotion Recall",
            "emotion_label_f1": "Emotion F1",
            "tier_accuracy_on_positive": "Tier Accuracy",
        }
    )
    table = table.sort_values(
        ["Emotion F1", "Tier Accuracy", "C3 Code Acc", "model_order", "prompt_order"],
        ascending=[False, False, False, True, True],
    ).head(top_n)
    return table[["Model", "Prompt Strategy", *C3_DEEP_DIVE_METRICS]].reset_index(drop=True)


def summarize_metric_winners(df: pd.DataFrame, id_columns: list[str]) -> pd.DataFrame:
    metric_columns = [
        column
        for column in df.columns
        if pd.api.types.is_numeric_dtype(df[column]) and column not in {"Models Averaged"}
    ]
    rows = []
    for metric in metric_columns:
        best_value = df[metric].max()
        winners = (
            df.loc[df[metric] == best_value, id_columns]
            .astype(str)
            .agg(" | ".join, axis=1)
            .tolist()
        )
        rows.append({
            "Metric": metric,
            "Best Value": best_value,
            "Winner(s)": "; ".join(winners),
        })
    return pd.DataFrame(rows)


In [2]:
RUN_DIR = DEFAULT_RUN_DIR

main_results_df = build_main_results_table(RUN_DIR)
main_best_df = summarize_metric_winners(main_results_df, ["Model", "Prompt Strategy"])

strategy_df = build_prompt_strategy_table(RUN_DIR)
strategy_best_df = summarize_metric_winners(strategy_df, ["Prompt Strategy"])

family_df = build_family_comparison_table(RUN_DIR)
family_best_df = summarize_metric_winners(family_df, ["Model"])

c3_df = build_c3_deep_dive_table(RUN_DIR, top_n=3)
c3_best_df = summarize_metric_winners(c3_df, ["Model", "Prompt Strategy"])


## Main Results Table

Variant-level results with cleaned model and prompt labels.

In [3]:
display(main_results_df.round(3))
display(main_best_df.round(3))


metric_label,Model,Prompt Strategy,C0 Acc,C1 Acc,C2 Abs Acc,C2 Rel Acc,C3 Code Acc,C3 Emotion F1,C4 Acc
0,Gemma 3 4B,Zero-shot,0.246,0.040,0.680,0.060,0.280,0.373,0.540
1,Qwen 3 8B,Zero-shot,0.338,0.130,0.776,0.122,0.520,0.358,0.707
2,GPT-5-nano,Zero-shot,0.462,0.470,0.750,0.250,0.352,0.082,0.641
3,Gemma 3 4B,Few-shot,0.108,0.030,0.768,0.354,0.340,0.338,0.616
4,Qwen 3 8B,Few-shot,0.323,0.430,0.828,0.081,0.400,0.332,0.495
5,GPT-5-nano,Few-shot,0.338,0.370,0.860,0.310,0.350,0.381,0.710
6,GPT-5.4-mini,Few-shot,0.431,0.650,0.840,0.460,0.450,0.486,0.740
7,Gemma 3 4B,Retrieval few-shot (MiniLM),0.328,0.131,0.730,0.320,0.410,0.398,0.620
8,Gemma 3 4B,Retrieval few-shot (TF-IDF),0.281,0.072,0.810,0.290,0.330,0.362,0.535
9,Qwen 3 8B,Retrieval few-shot (MiniLM),0.422,0.465,0.840,0.230,0.430,0.406,0.740


,Metric,Best Value,Winner(s)
0,C0 Acc,0.477,GPT-5-nano | Retrieval few-shot (MiniLM)
1,C1 Acc,0.650,GPT-5.4-mini | Few-shot
2,C2 Abs Acc,0.900,GPT-5.4-mini | Retrieval few-shot (MiniLM)
3,C2 Rel Acc,0.500,GPT-5.4-mini | Retrieval few-shot (MiniLM)
4,C3 Code Acc,0.520,Qwen 3 8B | Zero-shot
5,C3 Emotion F1,0.486,GPT-5.4-mini | Few-shot
6,C4 Acc,0.750,GPT-5.4-mini | Retrieval few-shot (MiniLM)


## Prompt Strategy Comparison

Prompt-strategy means after collapsing duplicate retrieval backends for the same model.

In [4]:
display(strategy_df.round(3))
display(strategy_best_df.round(3))


,Prompt Strategy,Models Averaged,C0 Acc,C1 Acc,C2 Abs Acc,C2 Rel Acc,C3 Code Acc,C3 Emotion F1,C4 Acc
0,Zero-shot,3,0.349,0.213,0.735,0.144,0.384,0.271,0.629
1,Few-shot,4,0.300,0.370,0.824,0.301,0.385,0.384,0.640
2,Retrieval few-shot,4,0.404,0.400,0.844,0.352,0.406,0.409,0.691


,Metric,Best Value,Winner(s)
0,C0 Acc,0.404,Retrieval few-shot
1,C1 Acc,0.400,Retrieval few-shot
2,C2 Abs Acc,0.844,Retrieval few-shot
3,C2 Rel Acc,0.352,Retrieval few-shot
4,C3 Code Acc,0.406,Retrieval few-shot
5,C3 Emotion F1,0.409,Retrieval few-shot
6,C4 Acc,0.691,Retrieval few-shot


## Model Family Comparison

Model-level means across the prompt strategies available for each model.

In [5]:
display(family_df.round(3))
display(family_best_df.round(3))


,Model,Prompt Strategies,C0 Acc,C1 Acc,C2 Abs Acc,C2 Rel Acc,C3 Code Acc,C3 Emotion F1,C4 Acc
0,Gemma 3 4B,"Zero-shot, Few-shot, Retrieval few-shot",0.220,0.057,0.739,0.239,0.330,0.364,0.578
1,Qwen 3 8B,"Zero-shot, Few-shot, Retrieval few-shot",0.365,0.332,0.819,0.153,0.442,0.365,0.637
2,GPT-5-nano,"Zero-shot, Few-shot, Retrieval few-shot",0.421,0.430,0.820,0.302,0.367,0.277,0.692
3,GPT-5.4-mini,"Few-shot, Retrieval few-shot",0.423,0.630,0.870,0.480,0.450,0.485,0.745


,Metric,Best Value,Winner(s)
0,C0 Acc,0.423,GPT-5.4-mini
1,C1 Acc,0.630,GPT-5.4-mini
2,C2 Abs Acc,0.870,GPT-5.4-mini
3,C2 Rel Acc,0.480,GPT-5.4-mini
4,C3 Code Acc,0.450,GPT-5.4-mini
5,C3 Emotion F1,0.485,GPT-5.4-mini
6,C4 Acc,0.745,GPT-5.4-mini


## Optional C3 Deep Dive

Top variants ranked by emotion F1, plus the winner for each C3 sub-metric.

In [6]:
display(c3_df.round(3))
display(c3_best_df.round(3))


,Model,Prompt Strategy,C3 Code Acc,Emotion Precision,Emotion Recall,Emotion F1,Tier Accuracy
0,GPT-5.4-mini,Few-shot,0.450,0.415,0.586,0.486,0.910
1,GPT-5.4-mini,Retrieval few-shot (MiniLM),0.450,0.421,0.567,0.484,0.872
2,Qwen 3 8B,Retrieval few-shot (TF-IDF),0.380,0.324,0.548,0.407,0.910


,Metric,Best Value,Winner(s)
0,C3 Code Acc,0.450,GPT-5.4-mini | Few-shot; GPT-5.4-mini | Retrie...
1,Emotion Precision,0.421,GPT-5.4-mini | Retrieval few-shot (MiniLM)
2,Emotion Recall,0.586,GPT-5.4-mini | Few-shot
3,Emotion F1,0.486,GPT-5.4-mini | Few-shot
4,Tier Accuracy,0.910,GPT-5.4-mini | Few-shot; Qwen 3 8B | Retrieval...
